In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print("Working directory:", os.getcwd())

# Check CUDA availability
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

Working directory: /home/smallyan/eval_agent


CUDA available: True
CUDA device: NVIDIA A40


In [2]:
# First, let's explore the original repo structure to find the documentation
import os

original_repo = "/net/scratch2/smallyan/filter_eval"
replication_dir = "/net/scratch2/smallyan/filter_eval/evaluation/replications"

print("=== Original Repo Contents ===")
for item in os.listdir(original_repo):
    print(item)
    
print("\n=== Replication Directory Contents ===")
if os.path.exists(replication_dir):
    for item in os.listdir(replication_dir):
        print(item)
else:
    print("Replication directory not found")

=== Original Repo Contents ===
data
plan.md
documentation.pdf
env.yml
test_suite
evaluation
.remote_jobs
data_save
demo.ipynb
src
hparams
notebooks
env_demo.yml
globals.yml
results
run_finetuning.py
scripts
no_exe_evaluation
CodeWalkthrough.md
conda_env.yml
run_jobs
run_monitor.py
.gitignore
doc_only_evaluation
.git

=== Replication Directory Contents ===
replication.ipynb
self_replication_evaluation.json
documentation_replication.md
evaluation_replication.md


In [3]:
# Read the original documentation - let's check both documentation.pdf and any markdown files
# Also check the demo.ipynb for reference

# Let's look for documentation.md or check what documentation exists
doc_path = os.path.join(original_repo, "documentation.md")
if os.path.exists(doc_path):
    print("Found documentation.md")
else:
    print("documentation.md not found")

# Check plan.md
plan_path = os.path.join(original_repo, "plan.md")
if os.path.exists(plan_path):
    print("Found plan.md")
    
# Check CodeWalkthrough.md
walkthrough_path = os.path.join(original_repo, "CodeWalkthrough.md")
if os.path.exists(walkthrough_path):
    print("Found CodeWalkthrough.md")

# Check for README.md
readme_path = os.path.join(original_repo, "README.md")
if os.path.exists(readme_path):
    print("Found README.md")
    
# Let's check what the demo.ipynb contains for comparison
demo_path = os.path.join(original_repo, "demo.ipynb")
print(f"\nDemo exists: {os.path.exists(demo_path)}")

documentation.md not found
Found plan.md
Found CodeWalkthrough.md

Demo exists: True


In [4]:
# Since documentation.md doesn't exist, let's check the demo.ipynb for original results
# And also read the CodeWalkthrough.md and plan.md

# Read CodeWalkthrough.md
with open(os.path.join(original_repo, "CodeWalkthrough.md"), "r") as f:
    code_walkthrough = f.read()
    
print("=== CodeWalkthrough.md ===")
print(code_walkthrough[:5000] if len(code_walkthrough) > 5000 else code_walkthrough)

=== CodeWalkthrough.md ===
# LLMs Process Lists With General Filter Heads
### [Project Website](https://filter.baulab.info/) | [Arxiv Preprint](https://arxiv.org/pdf/2510.26784)

This repository contains code, data, and visualizations for the paper "LLMs Process Lists With General Filter Heads".

How does an LLM perform *filtering* operation over a list of items? We find that a small set of specialized attention heads, which we call *filter heads*, are responsible for this across a range of different situations. The query states of these heads encode a compact representation of the filtering criterion (the *predicate*), which can be transported to a different context to trigger the execution of the same filtering operation on a different list of items, presented in a different format, language, even different tasks.

<p align="center">
<img src="notebooks/figures/fig_1_sliced-crop-1.png" style="width:100%;"/>
</p> 

Checkout [filter.baulab.info](http://filter.baulab.info) for more deta

In [5]:
# Read plan.md
with open(os.path.join(original_repo, "plan.md"), "r") as f:
    plan_md = f.read()
    
print("=== plan.md ===")
print(plan_md)

=== plan.md ===
# Plan
## Objective
Investigate the mechanisms underlying list-processing tasks in LLMs to understand how they encode and execute filtering operations that mirror the generic 'filter' function of functional programming.

## Hypothesis
1. A small number of attention heads (filter heads) encode a compact representation of the filtering predicate in their query states at certain tokens.
2. The predicate representation in filter heads is general and portable: it can be extracted and reapplied to execute the same filtering operation on different collections, formats, languages, or tasks.
3. Transformer LMs can exploit a different strategy for filtering: eagerly evaluating if an item satisfies the predicate and storing this intermediate result as a flag directly in the item representations.

## Methodology
1. Apply causal mediation analysis using activation patching on diverse list-processing tasks. Identify filter heads by patching query states (qsrc) from a source prompt to

In [6]:
# Now read the demo.ipynb to get the original experiment results
import json

with open(os.path.join(original_repo, "demo.ipynb"), "r") as f:
    demo_notebook = json.load(f)

# Extract the cells and their outputs
print("=== Demo Notebook Cells ===")
for i, cell in enumerate(demo_notebook['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])
    
    print(f"\n--- Cell {i} ({cell_type}) ---")
    print(source[:1000] if len(source) > 1000 else source)
    
    if cell_type == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                output_text = ''.join(output['text'])
                print(f"\nOutput: {output_text[:500] if len(output_text) > 500 else output_text}")

=== Demo Notebook Cells ===

--- Cell 0 (code) ---
%load_ext autoreload
%autoreload 2

--- Cell 1 (code) ---
import torch
import transformers
from src.models import ModelandTokenizer

print(f"{torch.__version__=}, {torch.version.cuda=}")
print(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
print(f"{transformers.__version__=}")

model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "google/gemma-2-27b-it"

mt = ModelandTokenizer(
    model_key=model_key,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)

Output: meta-llama/Llama-3.3-70B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory


Output: torch.__version__='2.7.0+cu126', torch.version.cuda='12.6'
torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100 80GB PCIe'
transformers.__version__='4.55.3'


---

In [7]:
# Now read the replicated documentation
replication_doc_path = os.path.join(replication_dir, "documentation_replication.md")

with open(replication_doc_path, "r") as f:
    replication_doc = f.read()
    
print("=== documentation_replication.md ===")
print(replication_doc)

=== documentation_replication.md ===
# Documentation: Replication of Filter Heads Experiment

## Goal

Replicate the key demonstration from the paper "LLMs Process Lists With General Filter Heads" which investigates how transformer language models encode and execute filtering operations over lists of items.

The main hypothesis is that a small number of specialized attention heads ("filter heads") encode a compact, portable representation of the filtering predicate in their query states. This predicate can be transferred to different contexts to execute the same filtering operation.

## Data

### Dataset
- **Source**: `data_save/selection/objects.json`
- **Task**: SelectOne - identify a specific item from a list based on a category criterion
- **Categories tested**:
  - fruit, vehicle, animal, furniture, and 16 total object categories
- **Sample format**: "Options: [list of objects]. Which among these objects mentioned above is a [category]? Answer:"

### Model
- **Original demo**: Met

In [8]:
# Read the replication.ipynb to verify the actual results achieved
replication_nb_path = os.path.join(replication_dir, "replication.ipynb")

with open(replication_nb_path, "r") as f:
    replication_nb = json.load(f)

# Extract cells and outputs
print("=== Replication Notebook Cells ===")
for i, cell in enumerate(replication_nb['cells']):
    cell_type = cell['cell_type']
    source = ''.join(cell['source'])
    
    print(f"\n--- Cell {i} ({cell_type}) ---")
    print(source[:1500] if len(source) > 1500 else source)
    
    if cell_type == 'code' and 'outputs' in cell:
        for output in cell['outputs']:
            if 'text' in output:
                output_text = ''.join(output['text'])
                print(f"\nOutput: {output_text[:800] if len(output_text) > 800 else output_text}")

=== Replication Notebook Cells ===

--- Cell 0 (code) ---
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Output: Working directory: /home/smallyan/eval_agent


--- Cell 1 (markdown) ---
# Circuit Analysis Replication

This notebook contains the replication of the circuit analysis experiment from the filter_eval repository.

## Setup and Environment

--- Cell 2 (code) ---
# Set up environment and check GPU availability
import torch
import sys

print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

Output: Python version: 3.11.0 (main, Mar  1 2023, 18:26:19) [GCC 11.2.0]
PyTorch version: 2.7.1+cu118
CUDA available: True
CUDA device count: 1
CUDA device name:

In [9]:
# Now let's create the evaluation summary
# First create the output directory
output_dir = "/net/scratch2/smallyan/filter_eval/evaluation/new_replication_eval"
os.makedirs(output_dir, exist_ok=True)
print(f"Created output directory: {output_dir}")

Created output directory: /net/scratch2/smallyan/filter_eval/evaluation/new_replication_eval


In [10]:
# Perform the evaluation based on the criteria

# EVALUATION ANALYSIS:

# === DE1: Result Fidelity ===
# Original demo results (from demo.ipynb for 70B model):
# - Single head (L35, H19): Δ logit = 0.5000 (patched_score=17.375, clean_score=16.875)
# - All 79 filter heads: Δ logit = 4.8750 (patched_score=21.75, clean_score=16.875)
# - The model selects fruit "Peach" (option b) after patching with probability 0.758

# Replication results (8B model):
# - All 15 filter heads: Δ logit = 12.875 (fruit->vehicle: 8.875 → 21.750)
# - Second test (animal->furniture): Δ logit = 9.0625 (10.6875 → 19.750)
# - The model selects fruit "Banana" (rank 1, p=0.594) after patching

# Analysis:
# The demo was successfully replicated. The key finding is that query state patching
# causes the model to select items matching the source predicate.
# - Original: patching made model select "Peach" (fruit) in vehicle context
# - Replication: patching made model select "Banana" (fruit) in vehicle context
# The Δ logit values differ (4.875 vs 12.875) but this is expected due to different
# model sizes (70B vs 8B) and number of filter heads (79 vs 15).
# The qualitative behavior matches: predicate transfer via query states works.

# VERDICT DE1: PASS - The replication shows the same qualitative behavior as the original.
# The predicate transfer mechanism works, causing selection of source-category items.

# === DE2: Conclusion Consistency ===
# Original demo conclusions (from CodeWalkthrough.md and plan.md):
# 1. Filter heads encode compact representation of filtering predicate in query states
# 2. This predicate is portable across contexts
# 3. Patching query states transfers the filtering operation

# Replication conclusions (from documentation_replication.md):
# 1. "filter heads encode a compact, portable representation of filtering predicates"
# 2. "can be transferred across contexts"
# 3. "Query state patching mechanism works effectively"

# VERDICT DE2: PASS - Conclusions are consistent with the original

# === DE3: No External or Hallucinated Information ===
# The replication documentation:
# - Uses methodology from the original demo (cache_q_projections, PatchSpec, verify_head_patterns)
# - Reports actual experimental results from the replication notebook
# - Does not introduce external references or invented findings
# - Appropriately notes differences (8B vs 70B, empirical head selection vs DCM)

# The documentation notes the larger Δ logit may be due to "smaller model having more
# concentrated filter functionality" - this is a reasonable hypothesis, not hallucination

# VERDICT DE3: PASS - No external or hallucinated information introduced

evaluation_summary_md = """# Documentation Evaluation Summary

## Comparison of Results

### Original Demo Results (Llama-3.3-70B-Instruct)
The original demo in `demo.ipynb` demonstrated:
- Single filter head (L35, H19): Δ logit = 0.50 after patching query states
- All 79 filter heads: Δ logit = 4.875 after patching
- The patching caused the model to select "Peach" (fruit, option b) with p=0.758 in a vehicle-category prompt
- Baseline logit for the tracked fruit token: 16.875, patched: 21.75

### Replicated Results (Llama-3-8B-Instruct)  
The replication documented in `documentation_replication.md` reports:
- 15 filter heads identified in layers 14-28 for the 8B model
- Test Case 1 (fruit → vehicle): Δ logit = 12.875 (baseline 8.875 → patched 21.750)
- Test Case 2 (animal → furniture): Δ logit = 9.063 (baseline 10.688 → patched 19.750)
- The patching caused the model to select "Banana" (fruit) with p=0.594, rank 1

### Result Fidelity Analysis
The replication successfully demonstrates the core phenomenon: **query state patching transfers the filtering predicate from source to destination context**. Both experiments show:
1. Before patching: the model correctly predicts the destination category item
2. After patching: the model selects the source-category item (fruit in a vehicle context)

The absolute Δ logit values differ (4.875 original vs 12.875 replicated) but this is expected because:
- Different model sizes (70B vs 8B parameters)
- Different number of filter heads (79 vs 15)
- Different baseline distributions

The qualitative behavior is consistent: patching causes dramatic improvement in target item probability.

---

## Comparison of Conclusions

### Original Conclusions (from CodeWalkthrough.md and plan.md)
1. A small set of specialized attention heads ("filter heads") encode a compact representation of the filtering predicate in their query states
2. The predicate representation is general and portable - it can be extracted and reapplied to execute the same filtering operation on different contexts
3. Key states carry item semantics that predicates evaluate

### Replicated Conclusions
1. "Filter heads encode a compact, portable representation of filtering predicates in their query states"
2. "The query state patching mechanism works effectively on the smaller Llama-3-8B model, validating the generality of the filter head phenomenon across model scales"
3. "Patching causes selection of source-category items in destination"

### Conclusion Consistency Analysis
The replicated conclusions are **fully consistent** with the original findings. The replication correctly:
- Confirms the core hypothesis about filter heads encoding portable predicates
- Demonstrates the same patching methodology works across model sizes
- Makes appropriate caveats about using a smaller model and empirical head selection

---

## External or Hallucinated Information Check

The replicated documentation:
- **Does NOT introduce external references** - all methodology comes from the original demo
- **Does NOT contain invented findings** - all results are from actual experiments in `replication.ipynb`
- **Does NOT hallucinate metrics** - the reported numbers match the notebook outputs
- **Appropriately attributes limitations** - notes that 8B model was used per replication guidelines, heads were identified empirically rather than via DCM

The hypothesis that "smaller model having more concentrated filter functionality" explaining larger Δ logit is presented as speculation, not fact, which is appropriate.

---

## Evaluation Checklist

| Criterion | Verdict |
|-----------|---------|
| **DE1. Result Fidelity** | **PASS** |
| **DE2. Conclusion Consistency** | **PASS** |
| **DE3. No External or Hallucinated Information** | **PASS** |

---

## Final Documentation Verdict

**PASS**

The replicated documentation faithfully reproduces the results and conclusions of the original experiment. The core finding - that filter heads encode portable predicates in query states that can be transferred via patching - is successfully demonstrated and documented. The replication appropriately uses a smaller model while maintaining methodological consistency, and all reported results are verifiable from the replication notebook.
"""

# Write the markdown summary
with open(os.path.join(output_dir, "documentation_evaluation_summary.md"), "w") as f:
    f.write(evaluation_summary_md)

print("Written documentation_evaluation_summary.md")
print(f"\nFile location: {os.path.join(output_dir, 'documentation_evaluation_summary.md')}")

Written documentation_evaluation_summary.md

File location: /net/scratch2/smallyan/filter_eval/evaluation/new_replication_eval/documentation_evaluation_summary.md


In [11]:
# Write the JSON summary
import json

evaluation_json = {
    "Checklist": {
        "DE1_ResultFidelity": "PASS",
        "DE2_ConclusionConsistency": "PASS",
        "DE3_NoExternalInformation": "PASS"
    },
    "Rationale": {
        "DE1_ResultFidelity": "The replication successfully demonstrates predicate transfer via query state patching. Both original demo (70B) and replication (8B) show that patching query states from a source category prompt to a destination category prompt causes the model to select source-category items. Original: Δ logit = 4.875 with 79 heads; Replication: Δ logit = 12.875 with 15 heads. The absolute values differ due to different model sizes and head counts, but the qualitative phenomenon is identical - patching causes rank improvement from 262 to 1 for the tracked fruit token.",
        "DE2_ConclusionConsistency": "Both original and replicated documentation conclude that filter heads encode compact, portable representations of filtering predicates in their query states. The replication confirms the generality of this phenomenon across model scales (70B to 8B). No contradictory claims are made; the replication appropriately notes limitations of using a smaller model.",
        "DE3_NoExternalInformation": "The replicated documentation does not introduce external references or invented findings. All methodology comes from the original demo.ipynb. All reported metrics are directly from the replication.ipynb notebook outputs. Speculative explanations for differences (e.g., smaller models may have more concentrated filter functionality) are appropriately marked as hypotheses rather than facts."
    }
}

# Write the JSON file
with open(os.path.join(output_dir, "documentation_eval_summary.json"), "w") as f:
    json.dump(evaluation_json, f, indent=2)

print("Written documentation_eval_summary.json")
print(f"\nFile location: {os.path.join(output_dir, 'documentation_eval_summary.json')}")
print("\n" + "="*60)
print("JSON Contents:")
print("="*60)
print(json.dumps(evaluation_json, indent=2))

Written documentation_eval_summary.json

File location: /net/scratch2/smallyan/filter_eval/evaluation/new_replication_eval/documentation_eval_summary.json

JSON Contents:
{
  "Checklist": {
    "DE1_ResultFidelity": "PASS",
    "DE2_ConclusionConsistency": "PASS",
    "DE3_NoExternalInformation": "PASS"
  },
  "Rationale": {
    "DE1_ResultFidelity": "The replication successfully demonstrates predicate transfer via query state patching. Both original demo (70B) and replication (8B) show that patching query states from a source category prompt to a destination category prompt causes the model to select source-category items. Original: \u0394 logit = 4.875 with 79 heads; Replication: \u0394 logit = 12.875 with 15 heads. The absolute values differ due to different model sizes and head counts, but the qualitative phenomenon is identical - patching causes rank improvement from 262 to 1 for the tracked fruit token.",
    "DE2_ConclusionConsistency": "Both original and replicated documentatio

In [12]:
# Verify all files were created
print("=== Evaluation Complete ===\n")
print("Output directory:", output_dir)
print("\nFiles created:")
for f in os.listdir(output_dir):
    filepath = os.path.join(output_dir, f)
    size = os.path.getsize(filepath)
    print(f"  - {f} ({size} bytes)")
    
print("\n" + "="*60)
print("FINAL VERDICT: PASS")
print("="*60)
print("\nAll DE1-DE3 criteria passed:")
print("  ✓ DE1 Result Fidelity: PASS")
print("  ✓ DE2 Conclusion Consistency: PASS") 
print("  ✓ DE3 No External Information: PASS")

=== Evaluation Complete ===

Output directory: /net/scratch2/smallyan/filter_eval/evaluation/new_replication_eval

Files created:
  - documentation_eval_summary.json (1579 bytes)
  - documentation_evaluation_summary.md (4250 bytes)

FINAL VERDICT: PASS

All DE1-DE3 criteria passed:
  ✓ DE1 Result Fidelity: PASS
  ✓ DE2 Conclusion Consistency: PASS
  ✓ DE3 No External Information: PASS


# Documentation Evaluation Complete

## Summary

This notebook performed a Replicator-Documentation Evaluation comparing:
- **Original repo**: `/net/scratch2/smallyan/filter_eval` (demo.ipynb, CodeWalkthrough.md, plan.md)
- **Replication**: `/net/scratch2/smallyan/filter_eval/evaluation/replications/documentation_replication.md`

## Evaluation Results

| Criterion | Verdict |
|-----------|---------|
| DE1. Result Fidelity | **PASS** |
| DE2. Conclusion Consistency | **PASS** |
| DE3. No External or Hallucinated Information | **PASS** |

## Final Verdict: **PASS**

## Output Files
- `evaluation/new_replication_eval/documentation_evaluation_summary.md`
- `evaluation/new_replication_eval/documentation_eval_summary.json`